In [ ]:
# =========================================================
# PIPELINE COMPLET: FEATURES TEMPORELLES + OPTUNA + EVALUATION MAP@12
# =========================================================

import pandas as pd
import numpy as np
import joblib
import pickle
import gc
import time
from collections import defaultdict
from typing import List, Dict, Set, Optional, Tuple
from dataclasses import dataclass
from tqdm import tqdm
import logging
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# =========================================================
# 1. UTILITAIRES TEMPORELS
# =========================================================

class TemporalFeatures:
    """Utilitaires pour features temporelles"""
    
    @staticmethod
    def get_month(date: pd.Timestamp) -> int:
        """Mois (1-12)"""
        return date.month
    
    @staticmethod
    def get_season(date: pd.Timestamp) -> int:
        """Saison: 0=Hiver, 1=Printemps, 2=Ete, 3=Automne"""
        month = date.month
        if month in [12, 1, 2]:
            return 0
        elif month in [3, 4, 5]:
            return 1
        elif month in [6, 7, 8]:
            return 2
        else:
            return 3
    
    @staticmethod
    def get_day_of_week(date: pd.Timestamp) -> int:
        """Jour de la semaine (0=Lundi, 6=Dimanche)"""
        return date.dayofweek


def create_temporal_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Cree features temporelles: day_of_week, mois, saison"""
    
    logger.info("Creation features temporelles...")
    df = transactions.copy()
    df['t_dat'] = pd.to_datetime(df['t_dat'])
    
    df['day_of_week'] = df['t_dat'].apply(TemporalFeatures.get_day_of_week)
    df['month'] = df['t_dat'].apply(TemporalFeatures.get_month)
    df['season'] = df['t_dat'].apply(TemporalFeatures.get_season)
    
    logger.info("[OK] Features temporelles creees")
    return df


def compute_user_temporal_features(transactions_with_temporal: pd.DataFrame) -> pd.DataFrame:
    """Calcule features temporelles par user"""
    
    logger.info("Agregation features temporelles par user...")
    
    user_temporal = transactions_with_temporal.groupby('customer_id').agg({
        'day_of_week': lambda x: x.mode()[0] if len(x.mode()) > 0 else 0,
        'month': lambda x: x.mode()[0] if len(x.mode()) > 0 else 0,
        'season': lambda x: x.mode()[0] if len(x.mode()) > 0 else 0
    })
    
    user_temporal.columns = [
        'user_preferred_day',
        'user_preferred_month',
        'user_preferred_season'
    ]
    
    logger.info(f"[OK] {user_temporal.shape[0]:,} users")
    return user_temporal


def compute_item_temporal_features(transactions_with_temporal: pd.DataFrame) -> pd.DataFrame:
    """Calcule features temporelles par item"""
    
    logger.info("Agregation features temporelles par item...")
    
    item_temporal = transactions_with_temporal.groupby('article_id').agg({
        'season': lambda x: x.mode()[0] if len(x.mode()) > 0 else 0,
        'month': lambda x: x.mode()[0] if len(x.mode()) > 0 else 0
    })
    
    item_temporal.columns = ['item_peak_season', 'item_peak_month']
    
    logger.info(f"[OK] {item_temporal.shape[0]:,} items")
    return item_temporal


def get_current_temporal_context(date: pd.Timestamp) -> Dict[str, int]:
    """Extrait contexte temporel pour une date"""
    return {
        'day_of_week': TemporalFeatures.get_day_of_week(date),
        'month': TemporalFeatures.get_month(date),
        'season': TemporalFeatures.get_season(date)
    }


# =========================================================
# 2. FONCTION MAP@12
# =========================================================

def map_at_k_fast(preds: Dict, actuals: Dict, k: int = 12) -> float:
    """Calcule MAP@k vectorise"""
    
    scores = []
    for user_id, recs in preds.items():
        actual = actuals.get(user_id, set())
        if not actual:
            continue
        
        hits = np.array([item in actual for item in recs[:k]])
        if not hits.any():
            scores.append(0.0)
            continue
        
        cumsum_hits = np.cumsum(hits)
        precisions = cumsum_hits * hits / np.arange(1, len(hits) + 1)
        scores.append(precisions.sum() / min(len(actual), k))
    
    return float(np.mean(scores)) if scores else 0.0


# =========================================================
# 3. FONCTION RECOMMANDATION BATCH
# =========================================================

def recommend_batch_lgbm(ranker, X_df, X, users, user_history_test, top_k=12):
    """
    Genere recommandations pour un batch d'users.
    Retourne dict {user_id: [item1, item2, ...]}
    """
    
    preds_dict = {}
    
    for user_id in tqdm(users, desc="Recommandations batch"):
        mask = X_df['user_id'] == user_id
        
        if not mask.any():
            continue
        
        df_subset = X_df[mask].copy()
        X_subset = X[mask]
        
        # Predire scores
        df_subset['score'] = ranker.predict(X_subset)
        
        # Trier et recuperer top-k
        top_items = (
            df_subset.sort_values('score', ascending=False)
            .head(top_k)['item_id']
            .tolist()
        )
        
        preds_dict[user_id] = top_items
    
    return preds_dict


# =========================================================
# 4. PIPELINE PRINCIPAL AVEC EVALUATION
# =========================================================

def evaluate_model_with_temporal(
    transactions_train,
    transactions_test,
    user_features_base,
    item_features_base,
    interaction_features,
    selected_features,
    user_history_train,
    user_history_test,
    als_cache,
    popular_items_idx,
    user_map,
    item_map,
    ALL_ITEMS,
    lgbm_params: Optional[Dict] = None,
    build_dataset_func = None
):
    """
    Pipeline complet: Features temporelles + Entrainement + Evaluation MAP@12
    
    Returns:
        map12_score: Score MAP@12 sur validation
        ranker: Modele entraine
        X_df_full: Dataset avec features
    """
    
    logger.info("=" * 70)
    logger.info("PIPELINE EVALUATION AVEC FEATURES TEMPORELLES")
    logger.info("=" * 70)
    
    # =============================================
    # ETAPE 1: PREPARATION FEATURES TEMPORELLES
    # =============================================
    
    logger.info("\n[ETAPE 1/5] Preparation features temporelles...")
    
    # Creer features temporelles
    transactions_train_temporal = create_temporal_features(transactions_train)
    
    # Agreger
    user_temporal_features = compute_user_temporal_features(transactions_train_temporal)
    item_temporal_features = compute_item_temporal_features(transactions_train_temporal)
    
    # Fusionner
    user_features = user_features_base.join(user_temporal_features, how='left')
    item_features = item_features_base.join(item_temporal_features, how='left')
    
    # Contexte temporel
    current_date = transactions_train['t_dat'].max()
    temporal_context = get_current_temporal_context(current_date)
    
    logger.info(f"[OK] User features: {user_features.shape}")
    logger.info(f"[OK] Item features: {item_features.shape}")
    logger.info(f"[OK] Contexte: {temporal_context}")
    
    # Ajouter features temporelles a selected_features
    temporal_features = [
        'user_preferred_day', 'user_preferred_month', 'user_preferred_season',
        'item_peak_season', 'item_peak_month',
        'context_day_of_week', 'context_month', 'context_season',
        'day_match_user', 'month_match_user', 'season_match_user',
        'season_match_item', 'month_match_item'
    ]
    
    selected_features_full = selected_features.copy()
    for feat in temporal_features:
        if feat not in selected_features_full:
            selected_features_full.append(feat)
    
    logger.info(f"[OK] Features totales: {len(selected_features_full)} (dont {len(temporal_features)} temporelles)")
    
    # =============================================
    # ETAPE 2: CONSTRUCTION DATASET
    # =============================================
    
    logger.info("\n[ETAPE 2/5] Construction dataset avec features temporelles...")
    
    all_users = list(user_history_train.keys())
    
    start_time = time.time()
    
    # Utiliser fonction build_dataset fournie
    X_df_full = build_dataset_func(
        users=all_users,
        user_features=user_features,
        item_features=item_features,
        user_temporal_features=user_temporal_features,
        item_temporal_features=item_temporal_features,
        interaction_features=interaction_features,
        user_history_train=user_history_train,
        user_history_test=user_history_test,
        als_cache=als_cache,
        popular_items_idx=popular_items_idx,
        item_map=item_map,
        ALL_ITEMS=ALL_ITEMS,
        selected_features=selected_features_full,
        temporal_context=temporal_context
    )
    
    elapsed = time.time() - start_time
    logger.info(f"[OK] Dataset cree en {elapsed:.1f}s")
    logger.info(f"     Shape: {X_df_full.shape}")
    
    # Preparation donnees
    y_full = X_df_full['label'].values
    X_full = X_df_full[selected_features_full].copy()
    group_full = X_df_full.groupby('user_id').size().values
    
    logger.info(f"[OK] X: {X_full.shape}, y: {y_full.shape} ({y_full.sum():,} positifs)")
    logger.info(f"     Groups: {len(group_full):,} users")
    
    # =============================================
    # ETAPE 3: ENTRAINEMENT LIGHTGBM
    # =============================================
    
    logger.info("\n[ETAPE 3/5] Entrainement LightGBM...")
    
    # Parametres par defaut si non fournis
    if lgbm_params is None:
        lgbm_params = {
            'objective': 'lambdarank',
            'metric': 'map',
            'eval_at': [12],
            'n_estimators': 300,
            'learning_rate': 0.05,
            'num_leaves': 63,
            'max_depth': 8,
            'min_child_samples': 20,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': 42,
            'n_jobs': -1,
            'force_row_wise': True,
            'verbose': -1
        }
    
    ranker = lgb.LGBMRanker(**lgbm_params)
    
    start_time = time.time()
    ranker.fit(X_full, y_full, group=group_full)
    elapsed = time.time() - start_time
    
    logger.info(f"[OK] Entrainement termine en {elapsed:.1f}s")
    
    # =============================================
    # ETAPE 4: GENERATION RECOMMANDATIONS
    # =============================================
    
    logger.info("\n[ETAPE 4/5] Generation recommandations...")
    
    start_time = time.time()
    preds_dict = recommend_batch_lgbm(
        ranker, X_df_full, X_full, all_users, user_history_test, top_k=12
    )
    elapsed = time.time() - start_time
    
    logger.info(f"[OK] Recommandations generees en {elapsed:.1f}s")
    logger.info(f"     {len(preds_dict):,} users")
    
    # =============================================
    # ETAPE 5: CALCUL MAP@12
    # =============================================
    
    logger.info("\n[ETAPE 5/5] Calcul MAP@12...")
    
    map12_score = map_at_k_fast(preds_dict, user_history_test, k=12)
    
    logger.info("\n" + "=" * 70)
    logger.info(f"RESULTAT FINAL: MAP@12 = {map12_score:.6f}")
    logger.info("=" * 70)
    
    return map12_score, ranker, X_df_full, selected_features_full


# =========================================================
# 5. OPTIMISATION HYPERPARAMETRES AVEC OPTUNA
# =========================================================

def optimize_hyperparameters_with_temporal(
    transactions_train,
    transactions_test,
    user_features_base,
    item_features_base,
    interaction_features,
    selected_features,
    user_history_train,
    user_history_test,
    als_cache,
    popular_items_idx,
    user_map,
    item_map,
    ALL_ITEMS,
    build_dataset_func,
    n_trials: int = 50
):
    """
    Optimise hyperparametres LightGBM avec Optuna.
    
    Returns:
        study: Etude Optuna complete
        best_ranker: Meilleur modele
        best_map12: Meilleur score MAP@12
    """
    
    logger.info("=" * 70)
    logger.info("OPTIMISATION HYPERPARAMETRES AVEC OPTUNA")
    logger.info("=" * 70)
    logger.info(f"Trials: {n_trials}")
    
    # Variables globales pour optimisation
    global_data = {
        'transactions_train': transactions_train,
        'transactions_test': transactions_test,
        'user_features_base': user_features_base,
        'item_features_base': item_features_base,
        'interaction_features': interaction_features,
        'selected_features': selected_features,
        'user_history_train': user_history_train,
        'user_history_test': user_history_test,
        'als_cache': als_cache,
        'popular_items_idx': popular_items_idx,
        'user_map': user_map,
        'item_map': item_map,
        'ALL_ITEMS': ALL_ITEMS,
        'build_dataset_func': build_dataset_func
    }
    
    def objective(trial):
        """Fonction objectif Optuna"""
        
        # Suggerer hyperparametres
        params = {
            'objective': 'lambdarank',
            'metric': 'map',
            'eval_at': [12],
            'random_state': 42,
            'n_jobs': -1,
            'force_row_wise': True,
            'verbose': -1,
            
            'num_leaves': trial.suggest_int('num_leaves', 31, 255),
            'max_depth': trial.suggest_int('max_depth', 6, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.5)
        }
        
        # Contrainte: num_leaves < 2^max_depth
        if params['num_leaves'] >= 2 ** params['max_depth']:
            params['num_leaves'] = 2 ** params['max_depth'] - 1
        
        # Evaluer avec ces parametres
        map12, _, _, _ = evaluate_model_with_temporal(
            lgbm_params=params,
            **global_data
        )
        
        return map12
    
    # Creer etude Optuna
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        study_name='h&m_lightgbm_temporal'
    )
    
    # Optimiser
    start_time = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    elapsed = time.time() - start_time
    
    # Resultats
    logger.info("\n" + "=" * 70)
    logger.info("RESULTATS OPTIMISATION")
    logger.info("=" * 70)
    logger.info(f"Temps total: {elapsed/60:.1f} minutes")
    logger.info(f"Trials completes: {len(study.trials)}")
    logger.info(f"Meilleur MAP@12: {study.best_value:.6f}")
    
    logger.info("\nMeilleurs hyperparametres:")
    for key, value in study.best_params.items():
        if isinstance(value, float):
            logger.info(f"  {key:25s}: {value:.4f}")
        else:
            logger.info(f"  {key:25s}: {value}")
    
    # Sauvegarder etude
    with open('optuna_study_temporal.pkl', 'wb') as f:
        pickle.dump(study, f)
    logger.info("\n[SAVE] optuna_study_temporal.pkl")
    
    # Entrainer modele final avec meilleurs params
    logger.info("\nEntrainement modele final avec meilleurs parametres...")
    
    best_params = {
        'objective': 'lambdarank',
        'metric': 'map',
        'eval_at': [12],
        'random_state': 42,
        'n_jobs': -1,
        'force_row_wise': True,
        'verbose': -1,
        **study.best_params
    }
    
    best_map12, best_ranker, _, _ = evaluate_model_with_temporal(
        lgbm_params=best_params,
        **global_data
    )
    
    # Sauvegarder meilleur modele
    joblib.dump(best_ranker, 'lightgbm_ranker_TUNED_temporal.pkl')
    logger.info("[SAVE] lightgbm_ranker_TUNED_temporal.pkl")
    
    return study, best_ranker, best_map12


# =========================================================
# 6. FONCTION PRINCIPALE
# =========================================================

def main_evaluate():
    """
    Point d'entree principal pour evaluation MAP@12.
    
    SANS optimisation: Evalue modele avec parametres par defaut
    AVEC optimisation: Utiliser main_optimize() a la place
    """
    
    logger.info("=" * 70)
    logger.info("MODE EVALUATION (SANS OPTIMISATION)")
    logger.info("=" * 70)
    
    # Evaluer avec parametres par defaut
    map12, ranker, X_df, features = evaluate_model_with_temporal(
        transactions_train=transactions_train,
        transactions_test=transactions_test,
        user_features_base=user_features,
        item_features_base=item_features,
        interaction_features=interaction_features,
        selected_features=selected_features,
        user_history_train=user_history_train,
        user_history_test=user_history_test,
        als_cache=als_cache,
        popular_items_idx=popular_items_idx,
        user_map=user_map,
        item_map=item_map,
        ALL_ITEMS=ALL_ITEMS,
        lgbm_params=None,  # Parametres par defaut
        build_dataset_func=build_dataset_optimized
    )
    
    # Sauvegarder modele
    joblib.dump(ranker, 'lightgbm_ranker_temporal.pkl')
    logger.info("\n[SAVE] lightgbm_ranker_temporal.pkl")
    
    return map12, ranker


def main_optimize(n_trials: int = 50):
    """
    Point d'entree principal pour optimisation hyperparametres + evaluation MAP@12.
    
    Args:
        n_trials: Nombre d'essais Optuna (30-50 recommande)
    
    Returns:
        study: Etude Optuna
        best_ranker: Meilleur modele
        best_map12: Meilleur score MAP@12
    """
    
    study, best_ranker, best_map12 = optimize_hyperparameters_with_temporal(
        transactions_train=transactions_train,
        transactions_test=transactions_test,
        user_features_base=user_features,
        item_features_base=item_features,
        interaction_features=interaction_features,
        selected_features=selected_features,
        user_history_train=user_history_train,
        user_history_test=user_history_test,
        als_cache=als_cache,
        popular_items_idx=popular_items_idx,
        user_map=user_map,
        item_map=item_map,
        ALL_ITEMS=ALL_ITEMS,
        build_dataset_func=build_dataset_optimized,
        n_trials=n_trials
    )
    
    return study, best_ranker, best_map12


# =========================================================
# 7. EXECUTION
# =========================================================

if __name__ == "__main__":
    
    # OPTION 1: Evaluation simple (RAPIDE - parametres par defaut)
    # map12, ranker = main_evaluate()
    
    # OPTION 2: Optimisation hyperparametres (LONG - 2-5h selon n_trials)
    study, best_ranker, best_map12 = main_optimize(n_trials=30)


In [1]:
import pandas as pd

# 1️⃣ Charger ta soumission existante
sub = pd.read_csv("submission_hybrid_cached.csv")

# Vérification rapide
print(sub.head())

# 2️⃣ Fonction pour corriger les article_id
def fix_prediction(pred):
    # pred = string "568601006 795440001 ..."
    return " ".join([str(a).zfill(10) for a in pred.split()])

# 3️⃣ Appliquer la correction
sub["prediction"] = sub["prediction"].apply(fix_prediction)

# 4️⃣ Vérification finale
print("Tous les article_id font 10 caractères ?",
      sub["prediction"].str.split().apply(lambda x: all(len(a)==10 for a in x)).all())

# 5️⃣ Sauvegarder le nouveau fichier
sub.to_csv("submission_hybrid_fixed.csv", index=False)

print("✅ Fichier sauvegardé : submission_hybrid_fixed.csv")


                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  568601043 898694001 568601044 568601023 568601...  
1  751471001 706016001 918292001 916468003 915526...  
2  794321007 872537004 891591001 915529003 915526...  
3  751471001 706016001 918292001 916468003 915526...  
4  791587001 730683050 927530004 896152002 791587...  
Tous les article_id font 10 caractères ? True
✅ Fichier sauvegardé : submission_hybrid_fixed.csv
